In [1]:
import pickle
import random

import torch

# Fix the kernel dying?
#import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

#from dollar_B import *

In [2]:
save_location = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\segfilt_rts_tensor_dict.pkl"

with open(save_location, 'rb') as f:
    full_dict = pickle.load(f)
    # I dont think this is actually tensors yet...
    tensor_dict = full_dict['data']

In [3]:
def ensure_channel_first(x: torch.Tensor) -> torch.Tensor:
    """Helper from eval_knn_proto.py to ensure (N, C, T) shape."""
    if x is None or x.dim() != 3:
        return x
    # If the last dimension matches known channel counts, swap it
    if x.shape[-1] in [16, 72]:
        return x.permute(0, 2, 1)
    return x

In [4]:
import os
import pickle
import random
import torch
from typing import Optional, List, Tuple

# =============================================================================
# 1. Episode Builder (Replaces your old build_episode)
# =============================================================================

def build_episode_dollar_B(
    user_data: dict, 
    k_shot: int, 
    n_way: int, 
    rng: random.Random,
    all_reps: List[int],
) -> Tuple[dict, dict]:
    """
    Extracts k_shot templates and the remaining query samples for n_way classes.
    Fixed split: first k_shot trials are support, the rest are query.
    """
    available_classes = [g for g in all_reps if g in user_data]
    chosen_classes = available_classes[:n_way]
    
    sup_emg, sup_imu, sup_labels = [], [], []
    qry_emg, qry_imu, qry_labels = [], [], []
    
    for c_idx, g in enumerate(chosen_classes):
        # NOTE: BY DEFAULT, data loads in as (N_trials, T, channels)
        emg_tensor = ensure_channel_first(user_data[g]["emg"])  # (N_trials, C_emg, T)
        imu_tensor = ensure_channel_first(user_data[g]["imu"])  # (N_trials, C_imu, T)
        
        # We assume trials are ordered. Take the first k_shot as templates.
        sup_emg.append(emg_tensor[:k_shot])
        sup_imu.append(imu_tensor[:k_shot])
        sup_labels.append(torch.full((k_shot,), c_idx, dtype=torch.long))
        
        # Take the rest as queries
        qry_emg.append(emg_tensor[k_shot:])
        qry_imu.append(imu_tensor[k_shot:])
        num_queries = emg_tensor.shape[0] - k_shot
        qry_labels.append(torch.full((num_queries,), c_idx, dtype=torch.long))
        
    support = {
        "emg": torch.cat(sup_emg, dim=0),
        "imu": torch.cat(sup_imu, dim=0),
        "labels": torch.cat(sup_labels, dim=0)
    }
    query = {
        "emg": torch.cat(qry_emg, dim=0),
        "imu": torch.cat(qry_imu, dim=0),
        "labels": torch.cat(qry_labels, dim=0)
    }
    return support, query



In [5]:
# TODO: New version: needs to be checked...
# all_reps renamed to available_gesture_classes with a docstring that makes the contract explicit. 
# --> I dont know that that is correct / what I want either...
# --> Maybe I was literally using all_reps as a makeshift available_gesture_classes too idfk
# --> I think the purpose of all_reps is so that we can do a train/test split based on gesture repetition (ie intra-subject) instead of by class or by user...
# --> Not sure if that is happening (was it ever happening with KNN lol) --> if the models are subj-sp, then we need a support/query split...
# rng.sample() is now actually used. 
# Added an assert that the stored enc_gesture_id matches the key used for lookup. 
# Returns gesture_ids (list of strings) in both support and query dicts — invaluable for debugging whether your episode actually contains the right classes.

def build_episode_dollar_B(
    user_data: dict,
    k_shot: int,
    n_way: int,
    rng: random.Random,
    available_gesture_classes: List[int],   # renamed: these are enc_gesture_ids (0..N-1)
) -> Tuple[dict, dict]:
    """
    Build a support/query episode for one user.
 
    FIXED vs original:
      - available_gesture_classes is now explicitly a list of enc_gesture_ids
        (0-indexed class labels), NOT repetition indices. Callers must pass
        the right thing; the name makes the contract clear.
      - rng is now actually used to randomly sample n_way classes each episode.
      - Labels (c_idx) are still relative (0..n_way-1) within the episode —
        this is correct for episodic/meta-learning and also fine for KNN
        (distances don't depend on label magnitude).
      - An optional 'gesture_ids' field is returned for debugging.
 
    Args:
        user_data:                data_dict[pid]  — keyed by enc_gesture_id
        k_shot:                   number of support trials per class
        n_way:                    number of classes per episode
        rng:                      random.Random instance (NOW USED)
        available_gesture_classes: list of enc_gesture_ids to sample from
                                   (e.g. train_gesture_ids = [0,1,3,5,7,8])
 
    Returns:
        support, query dicts each with keys: emg, imu, labels, gesture_ids
    """
    # Filter to classes that actually exist for this user
    valid_classes = [g for g in available_gesture_classes if g in user_data]
 
    if len(valid_classes) < n_way:
        raise ValueError(
            f"User only has {len(valid_classes)} valid classes "
            f"({valid_classes}), but n_way={n_way} requested."
        )
 
    # ✅ NOW uses rng — different classes sampled each episode
    chosen_classes = rng.sample(valid_classes, n_way)
 
    sup_emg, sup_imu, sup_labels, sup_gesture_ids = [], [], [], []
    qry_emg, qry_imu, qry_labels, qry_gesture_ids = [], [], [], []
 
    for c_idx, enc_gid in enumerate(chosen_classes):
        entry = user_data[enc_gid]
 
        # Sanity check: stored label should match the key we used
        assert entry['enc_gesture_id'] == enc_gid, (
            f"Label mismatch in user_data: key={enc_gid} but "
            f"stored enc_gesture_id={entry['enc_gesture_id']}"
        )
 
        emg_tensor = ensure_channel_first(entry['emg'])  # (N_reps, C_emg, T)
        imu_tensor = ensure_channel_first(entry['imu'])  # (N_reps, C_imu, T)
 
        n_reps = emg_tensor.shape[0]
        if n_reps <= k_shot:
            raise ValueError(
                f"enc_gesture_id {enc_gid} has only {n_reps} reps, "
                f"need > {k_shot} for k_shot={k_shot}."
            )
 
        # Support: first k_shot repetitions
        sup_emg.append(emg_tensor[:k_shot])
        sup_imu.append(imu_tensor[:k_shot])
        sup_labels.append(torch.full((k_shot,), c_idx, dtype=torch.long))
        sup_gesture_ids.extend([entry['gesture_id']] * k_shot)
 
        # Query: remaining repetitions
        n_query = n_reps - k_shot
        qry_emg.append(emg_tensor[k_shot:])
        qry_imu.append(imu_tensor[k_shot:])
        qry_labels.append(torch.full((n_query,), c_idx, dtype=torch.long))
        qry_gesture_ids.extend([entry['gesture_id']] * n_query)
 
    support = {
        "emg":         torch.cat(sup_emg, dim=0),   # (n_way*k_shot, C_emg, T)
        "imu":         torch.cat(sup_imu, dim=0),
        "labels":      torch.cat(sup_labels, dim=0),
        "gesture_ids": sup_gesture_ids,              # list of strings, for debugging
    }
    query = {
        "emg":         torch.cat(qry_emg, dim=0),
        "imu":         torch.cat(qry_imu, dim=0),
        "labels":      torch.cat(qry_labels, dim=0),
        "gesture_ids": qry_gesture_ids,
    }
    return support, query


In [8]:
# =============================================================================
# 2. The Exact $B Classifier 
# =============================================================================

import torch
import numpy as np
from sklearn.decomposition import PCA as sklearnPCA

def exact_dollar_B_classify(
    sup_emg: torch.Tensor,   
    sup_imu: torch.Tensor, 
    sup_labels: torch.Tensor,   
    qry_emg: torch.Tensor,   
    qry_imu: torch.Tensor, 
    n_components: int = 50,
    debug: bool = False,
    use_sklearn_pca: bool = False
) -> torch.Tensor:
    
    # 1. Early Fusion (Shared Space)
    sup = torch.cat([sup_emg, sup_imu], dim=1)  # (N_sup, C, T)
    qry = torch.cat([qry_emg, qry_imu], dim=1)  # (N_qry, C, T)

    N_sup, C, T = sup.shape
    N_qry = qry.shape[0]
    n_pc = min(n_components, C - 1)
    
    all_dists = torch.zeros(N_qry, N_sup, device=qry.device)

    if debug:
        print(f"\n--- DEBUG START ---")
        print(f"Total Channels (C): {C}, Time Steps (T): {T}")
        print(f"Concat Support Shape: {sup.shape}")
        print(f"Concat Query Shape:   {qry.shape}")

    # 2. Per-Template Loop
    for i in range(N_sup):
        template_ts = sup[i]  # (C, T)
        
        # [DEBUG] Check mean before subtraction
        raw_mean = template_ts.mean(dim=1)
        if debug and i == 0: # Only print for the first template to avoid spam
            print(f"\n[Template 0] Raw Channel Means (first 5): {raw_mean[:5].tolist()}")
            print(f"[Template 0] Shape BEFORE PCA: {template_ts.shape}")

        if use_sklearn_pca:
            # sklearn expects (n_samples, n_features) -> (T, C)
            t_np = template_ts.t().cpu().numpy()
            pca = sklearnPCA(n_components=n_pc)
            # Project template into its own space
            proj_t_raw = torch.from_numpy(pca.fit_transform(t_np)).t().to(sup.device) # (n_pc, T)
            # In sklearn, the components and mean are stored in the object
            U_t = torch.from_numpy(pca.components_).t().to(sup.device) # (C, n_pc)
            mean_c = torch.from_numpy(pca.mean_).unsqueeze(1).to(sup.device) # (C, 1)
        else:
            # Manual Torch Logic
            mean_c = template_ts.mean(dim=1, keepdim=True)
            x_centered = template_ts - mean_c
            cov = (x_centered @ x_centered.t()) / (T - 1)
            _, vecs = torch.linalg.eigh(cov)
            U_t = vecs[:, -n_pc:].flip(dims=[1])
            proj_t_raw = U_t.t() @ x_centered # (n_pc, T)

        proj_t = proj_t_raw.flatten() # (n_pc * T)

        if debug and i == 0:
            print(f"[Template 0] Shape AFTER PCA:  {proj_t_raw.shape} (Flattened: {proj_t.shape[0]})")

        # 3. Project all queries into THIS template's space
        for q in range(N_qry):
            xq_centered = qry[q] - mean_c
            proj_q_raw = U_t.t() @ xq_centered
            proj_q = proj_q_raw.flatten()
            
            if debug and i == 0 and q == 0:
                print(f"[Query 0 vs Template 0] Shape BEFORE PCA: {qry[q].shape}")
                print(f"[Query 0 vs Template 0] Shape AFTER PCA:  {proj_q_raw.shape}")

            all_dists[q, i] = (proj_q - proj_t).abs().sum()

    if debug: print("--- DEBUG END ---\n")

    best_template_idx = all_dists.argmin(dim=1)
    return sup_labels[best_template_idx]


# =============================================================================
# 3. Evaluator Pipeline
# =============================================================================

def run_dollar_B_evaluation(
    tensor_dict: dict,
    k_shot: int,
    n_way: int = 10,
    n_components: int = 50,
    seed: int = 42, 
    debug = False,
    use_sklearn_pca = False,
):
    print(f"\n{'='*65}")
    print(f"  EXACT $B EVAL: {k_shot}-shot  {n_way}-way")
    print(f"  n_components : {n_components} (EMG+IMU Early Fusion)")
    print(f"{'='*65}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rng = random.Random(seed)
    
    # Extract list of all possible reps/gestures across the dataset
    # We grab this from the first user's dictionary keys
    first_pid = list(tensor_dict.keys())[0]
    all_enc_class_labels = list(tensor_dict[first_pid].keys()) 
    
    accs = []
    
    for pid, user_data in tensor_dict.items():
        available = [g for g in all_enc_class_labels if g in user_data]
        if len(available) < n_way: 
            print("Skipping, not enough classes in this user!")
            continue # Skip if user doesn't have enough classes
            
        # Build episode for this user
        support, query = build_episode_dollar_B(
            user_data=user_data, 
            k_shot=k_shot, 
            n_way=n_way, 
            rng=rng, 
            available_gesture_classes=available
        )

        sup_emg = support["emg"].to(device)
        qry_emg = query["emg"].to(device)
        sup_imu = support["imu"].to(device)
        qry_imu = query["imu"].to(device)
        sup_labels = support["labels"].to(device)
        qry_labels = query["labels"].to(device)
        
        # Skip if they don't have enough shots
        if sup_emg.size(0) < k_shot * n_way:
            print("Skipping, out of shots!")
            continue

        # Run Prediction
        preds = exact_dollar_B_classify(
            sup_emg=sup_emg, 
            sup_imu=sup_imu, 
            sup_labels=sup_labels,
            qry_emg=qry_emg, 
            qry_imu=qry_imu,
            n_components=n_components, 
            debug=debug,
            use_sklearn_pca=use_sklearn_pca,
        )
        
        # Calculate Accuracy
        acc = (preds == qry_labels).float().mean().item()
        accs.append(acc)
        
        print(f"  PID {str(pid):>4} | exact_$B = {acc*100:>5.1f}%  [sup={sup_emg.size(0)}, qry={qry_emg.size(0)}]")

    if len(accs) > 0:
        mean_acc = torch.tensor(accs).mean().item()
        std_acc  = torch.tensor(accs).std().item()

        print(f"\n  {'Method':<30}  {'Mean':>8}  {'Std':>7}  {'Users':>6}")
        print(f"  {'─'*30}  {'─'*8}  {'─'*7}  {'─'*6}")
        print(f"  {'Exact $B ($B-faithful)':<30}  {mean_acc*100:>7.2f}%  {std_acc*100:>6.2f}%  {len(accs):>6}")
    else:
        print("No valid users found to evaluate.")


In [9]:

# Run for 1-shot (and 5-shot if you like)
for shots in [1, 5]:
    run_dollar_B_evaluation(
        tensor_dict=tensor_dict,
        k_shot=shots,
        n_way=10,            # 10-class problem
        n_components=50,     # Enforce the shared 50 PCs
        seed=42, 
        debug=True,
        use_sklearn_pca=False,
    )


  EXACT $B EVAL: 1-shot  10-way
  n_components : 50 (EMG+IMU Early Fusion)

--- DEBUG START ---
Total Channels (C): 88, Time Steps (T): 64
Concat Support Shape: torch.Size([10, 88, 64])
Concat Query Shape:   torch.Size([90, 88, 64])

[Template 0] Raw Channel Means (first 5): [0.0, -7.450580596923828e-09, 7.450580596923828e-09, -9.313225746154785e-10, -2.7939677238464355e-09]
[Template 0] Shape BEFORE PCA: torch.Size([88, 64])
[Template 0] Shape AFTER PCA:  torch.Size([50, 64]) (Flattened: 3200)
[Query 0 vs Template 0] Shape BEFORE PCA: torch.Size([88, 64])
[Query 0 vs Template 0] Shape AFTER PCA:  torch.Size([50, 64])
--- DEBUG END ---

  PID P102 | exact_$B =  80.0%  [sup=10, qry=90]

--- DEBUG START ---
Total Channels (C): 88, Time Steps (T): 64
Concat Support Shape: torch.Size([10, 88, 64])
Concat Query Shape:   torch.Size([90, 88, 64])

[Template 0] Raw Channel Means (first 5): [0.0, 0.0, 0.0, -3.725290298461914e-09, -1.862645149230957e-09]
[Template 0] Shape BEFORE PCA: torch.Siz

In [10]:

# Run for 1-shot (and 5-shot if you like)
for shots in [1, 5]:
    run_dollar_B_evaluation(
        tensor_dict=tensor_dict,
        k_shot=shots,
        n_way=10,            # 10-class problem
        n_components=50,     # Enforce the shared 50 PCs
        seed=42, 
        debug=True,
        use_sklearn_pca=True,
    )


  EXACT $B EVAL: 1-shot  10-way
  n_components : 50 (EMG+IMU Early Fusion)

--- DEBUG START ---
Total Channels (C): 88, Time Steps (T): 64
Concat Support Shape: torch.Size([10, 88, 64])
Concat Query Shape:   torch.Size([90, 88, 64])

[Template 0] Raw Channel Means (first 5): [0.0, -7.450580596923828e-09, 7.450580596923828e-09, -9.313225746154785e-10, -2.7939677238464355e-09]
[Template 0] Shape BEFORE PCA: torch.Size([88, 64])
[Template 0] Shape AFTER PCA:  torch.Size([50, 64]) (Flattened: 3200)
[Query 0 vs Template 0] Shape BEFORE PCA: torch.Size([88, 64])
[Query 0 vs Template 0] Shape AFTER PCA:  torch.Size([50, 64])
--- DEBUG END ---

  PID P102 | exact_$B =  80.0%  [sup=10, qry=90]

--- DEBUG START ---
Total Channels (C): 88, Time Steps (T): 64
Concat Support Shape: torch.Size([10, 88, 64])
Concat Query Shape:   torch.Size([90, 88, 64])

[Template 0] Raw Channel Means (first 5): [0.0, 0.0, 0.0, -3.725290298461914e-09, -1.862645149230957e-09]
[Template 0] Shape BEFORE PCA: torch.Siz